# 1 - Preset voices (Qwen3-TTS 1.7B CustomVoice)

**Locked production setup: speaker vivian, 1.7B, whole page, no direction.**

Two things were tested and rejected, so they are deliberately absent here:
sentence-by-sentence generation (each sentence loses the previous one's prosody and
gets its own attitude) and text normalisation (no audible difference).

Run top to bottom. The model load takes ~40 s; after that a page is ~0.6x real time
on the 4060.

In [1]:
# ============================ DEVICE ============================
# Choose the GPU BEFORE torch is imported.
#   "1"   -> RTX 4060   (safe: leaves the 4090 to ComfyUI)
#   "0"   -> RTX 4090   (fast, but only when ComfyUI is idle)
#   ""    -> CPU only   (very slow; works with no free GPU)
#
# The production voice worker also lives on GPU 1. While it runs there is only
# ~2.3 GB free there, which is NOT enough for the 1.7B model (~4.2 GB).
# Either stop it:  cd ~/ai_voice && docker compose down
# or set GPU = "0".
# Driver here is 615.71.09 / CUDA 13.4 -> install the cu130 wheels
# (see requirements.txt). Older cu12x wheels also run, because the
# driver is backward compatible.
import os
GPU = "1"

os.environ["CUDA_VISIBLE_DEVICES"] = GPU
DEVICE = "cuda:0" if GPU != "" else "cpu"   # after masking, the chosen card is always cuda:0
print("CUDA_VISIBLE_DEVICES =", GPU or "(none - CPU)")

CUDA_VISIBLE_DEVICES = 1


In [2]:
import time, json, subprocess
import numpy as np, torch, soundfile as sf
from IPython.display import Audio, display
from qwen_tts.inference.qwen3_tts_model import Qwen3TTSModel

MODELS = "/home/pavlo/lab/TTS/models"
OUT    = "/home/pavlo/lab/TTS/out"
os.makedirs(OUT, exist_ok=True)

def gpu_free():
    if DEVICE == "cpu": return "cpu"
    return subprocess.run(["nvidia-smi","--query-gpu=index,name,memory.free","--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip()
print(gpu_free())

SoX could not be found!

    If you do not have SoX, proceed here:
     - - - http://sox.sourceforge.net/ - - -

    If you do (or think that you should) have SoX, double-check your
    path variables.
    
/bin/sh: 1: sox: not found



********
********
 
0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 7885 MiB


In [3]:
# Loudness. Production target is -16 LUFS integrated, -1 dBTP ceiling, so narration
# sits predictably under the LTX ambient audio. Set NORMALISE = False for raw output.
NORMALISE   = True
TARGET_LUFS = -16.0

def normalise(a, sr):
    if not NORMALISE: return a
    try:
        import pyloudnorm as pyln
    except ImportError:
        print("pyloudnorm not installed - skipping"); return a
    m = pyln.Meter(sr)
    y = a * (10 ** ((TARGET_LUFS - m.integrated_loudness(a)) / 20.0))
    ceiling = 10 ** (-1.0/20.0)
    pk = float(np.max(np.abs(y)))
    if pk > ceiling: y = y * (ceiling / pk)
    return y.astype(np.float32)

def play(a, sr, name=None):
    a = normalise(np.asarray(a, dtype=np.float32), sr)
    if name:
        p = os.path.join(OUT, name if name.endswith(".wav") else name + ".wav")
        sf.write(p, a, sr, subtype="PCM_16"); print("saved", p)
    display(Audio(a, rate=sr))
    return a

## The nine preset voices

Median pitch, and how long each took to read one fixed
test page. Pace varies about 28% between them, which affects GPU cost as well as sound.

In [4]:
VOICES = {
    "dylan":    "167 Hz - lowest, fastest (15.8s)",
    "aiden":    "179 Hz - male, mid",
    "ryan":     "185 Hz - male, mid",
    "uncle_fu": "226 Hz",
    "vivian":   "231 Hz - *** PRODUCTION CHOICE ***",
    "ono_anna": "247 Hz - brisk (16.3s)",
    "serena":   "267 Hz",
    "sohee":    "267 Hz - brisk (16.4s)",
    "eric":     "276 Hz - highest and slowest (20.2s)",
}
for k, v in VOICES.items(): print(f"{k:<10} {v}")

dylan      167 Hz - lowest, fastest (15.8s)
aiden      179 Hz - male, mid
ryan       185 Hz - male, mid
uncle_fu   226 Hz
vivian     231 Hz - *** PRODUCTION CHOICE ***
ono_anna   247 Hz - brisk (16.3s)
serena     267 Hz
sohee      267 Hz - brisk (16.4s)
eric       276 Hz - highest and slowest (20.2s)


## Load the model

Run once. Re-run only if you change the device.

In [5]:
MODEL = MODELS + "/qwen3tts-17b-custom"     # 1.7B CustomVoice, ~4.2 GB VRAM
# MODEL = MODELS + "/qwen3tts-06b-custom"   # 0.6B, half the VRAM, ~13% faster

t0 = time.time()
model = Qwen3TTSModel.from_pretrained(
    MODEL, device_map=DEVICE,
    dtype=torch.bfloat16 if DEVICE != "cpu" else torch.float32)
print(f"loaded in {time.time()-t0:.0f}s")
print("speakers:", sorted(model.get_supported_speakers() or []))

loaded in 1s
speakers: ['aiden', 'dylan', 'eric', 'ono_anna', 'ryan', 'serena', 'sohee', 'uncle_fu', 'vivian']


## Generate

LANGUAGE takes the full word, not an ISO code: english, spanish, french, german,
italian, portuguese, russian, japanese, korean, chinese.

INSTRUCT is the optional narration direction. **Production ships with it empty** - in a
seeded A/B it made no audible difference once whole-page generation was used. It is here
to play with. Describe delivery, not plot.

In [6]:
SPEAKER  = "vivian"
LANGUAGE = "english"
SEED     = 1234          # same seed + same settings = same audio. None for random.
INSTRUCT = ""            # e.g. "Hushed and drowsy, bedtime register. Very gentle."

TEXT = """As the stars twinkled in the night sky, Elsa and Mickey sat by their castle,
dreaming of their next adventure. Together, they could face anything!"""

if SEED is not None:
    torch.manual_seed(SEED); np.random.seed(SEED)

t0 = time.time()
wavs, sr = model.generate_custom_voice(
    text=" ".join(TEXT.split()),     # ONE call per page - never split by sentence
    speaker=SPEAKER, language=LANGUAGE, instruct=INSTRUCT or "")
a = np.asarray(wavs[0], dtype=np.float32)
gen = time.time() - t0
print(f"{len(a)/sr:.2f}s audio in {gen:.2f}s (RTF {gen/(len(a)/sr):.3f}) sr={sr}")
_ = play(a, sr, name=f"{SPEAKER}_{LANGUAGE}")

Setting `pad_token_id` to `eos_token_id`:2150 for open-end generation.


9.84s audio in 6.57s (RTF 0.667) sr=24000
saved /home/pavlo/lab/TTS/out/vivian_english.wav


## Compare several voices on the same text

In [ ]:
TRY = ["vivian", "sohee", "serena", "ono_anna"]

for spk in TRY:
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    wavs, sr = model.generate_custom_voice(text=" ".join(TEXT.split()), speaker=spk,
                                           language=LANGUAGE, instruct=INSTRUCT or "")
    print("---", spk, VOICES.get(spk, ""))
    play(np.asarray(wavs[0], dtype=np.float32), sr, name="cmp_" + spk)

## Read a whole story

Paste the text of each page. One call per page, which is how
production will do it. Pages are saved individually and as one joined file.

In [ ]:
PAGES = [
    "One sunny day at the beach, Elsa and her friend Mickey decided to build the biggest sand castle ever!",
    "As they finished their castle, a tiny pirate ship appeared on the horizon!",
]

full = []
for i, page in enumerate(PAGES, 1):
    if SEED is not None:
        torch.manual_seed(SEED); np.random.seed(SEED)
    t0 = time.time()
    wavs, sr = model.generate_custom_voice(text=" ".join(page.split()), speaker=SPEAKER,
                                           language=LANGUAGE, instruct=INSTRUCT or "")
    a = normalise(np.asarray(wavs[0], dtype=np.float32), sr)
    sf.write(f"{OUT}/page_{i:02d}.wav", a, sr, subtype="PCM_16")
    full.append(a); full.append(np.zeros(int(sr*0.4), dtype=np.float32))
    print(f"page {i:02d}  {len(a)/sr:5.2f}s  gen {time.time()-t0:5.2f}s")

story = np.concatenate(full)
sf.write(f"{OUT}/story_full.wav", story, sr, subtype="PCM_16")
print(f"total {len(story)/sr:.1f}s -> {OUT}/story_full.wav")
display(Audio(story, rate=sr))

## Free the GPU

Run before loading another model, or before restarting the voice worker.

In [7]:
try:
    del model
except NameError:
    pass
torch.cuda.empty_cache()
print(gpu_free())

0, NVIDIA GeForce RTX 4090, 5841 MiB
1, NVIDIA GeForce RTX 4060, 7426 MiB
